# Korean Stops Production - Collective Analysis

This notebook creates visualizations from the preprocessed production data.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11

print("Libraries imported successfully")

## Load Preprocessed Data

In [ ]:
# Load preprocessed data
data_path = Path.cwd().parent / "shared" / "output" / "production_preprocessed_data.csv"
df = pd.read_csv(data_path, encoding='utf-8-sig')

print(f"Loaded {len(df)} rows from preprocessed data")
print(f"Participants: {df['prolific_id'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPhonation distribution:")
print(df['phonation'].value_counts())

## VOT vs F0 Visualization

In [ ]:
# Define visual settings for phonation types
phonation_order = ['aspirated', 'fortis', 'lenis']
phonation_colors = {
    'lenis': '#6baed6',      # Blue
    'fortis': '#fd8d3c',     # Orange
    'aspirated': '#74c476'   # Green
}
phonation_markers = {
    'lenis': 'o',       # Circle
    'fortis': 's',      # Square
    'aspirated': '^'    # Triangle
}
phonation_labels = {
    'lenis': 'lenis',
    'fortis': 'fortis',
    'aspirated': 'aspirated'
}

print("Visual settings configured")

In [ ]:
# Create scatter plot: Scaled VOT vs Normed F0
fig, ax = plt.subplots(figsize=(8, 8))

for phonation in phonation_order:
    df_phon = df[df['phonation'] == phonation]
    ax.scatter(
        df_phon['scaled_vot'], 
        df_phon['normed_f0'],
        c=phonation_colors[phonation], 
        marker=phonation_markers[phonation],
        s=60, 
        alpha=0.5, 
        label=phonation_labels[phonation],
        edgecolors='white',
        linewidth=0.5
    )

ax.set_title('VOT vs F0 (by phonation type)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Scaled VOT (word duration)', fontsize=12)
ax.set_ylabel('Normed F0 (z-score)', fontsize=12)
ax.legend(title='Phonation', fontsize=10, title_fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()

# Save figure
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "vot_vs_f0_by_phonation.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {output_path}")

plt.show()

# Print statistics
print(f"\nData summary:")
print(f"Total data points: {len(df)}")
for phonation in phonation_order:
    count = len(df[df['phonation'] == phonation])
    print(f"  {phonation}: {count} ({count/len(df)*100:.1f}%)")

## Violin Plot: Phonation Type vs Log Frequency

In [ ]:
# Check log_morpheme_freq statistics by phonation
print(f"\nLog morpheme frequency statistics:")
print(df.groupby('phonation')['log_morpheme_freq'].describe())

In [ ]:
# Create violin plot: Phonation Type vs Log Frequency
fig, ax = plt.subplots(figsize=(8, 8))

# Create violin plot with box plot overlay
parts = ax.violinplot(
    [df[df['phonation'] == phon]['log_morpheme_freq'].dropna()
     for phon in phonation_order],
    positions=[0, 1, 2],
    showmeans=False,
    showmedians=False,
    showextrema=False
)

# Color the violin plots
colors_list = [phonation_colors[phon] for phon in phonation_order]
for i, (pc, color) in enumerate(zip(parts['bodies'], colors_list)):
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_linewidth(1.5)
    pc.set_alpha(0.7)

# Add box plots on top
bp = ax.boxplot(
    [df[df['phonation'] == phon]['log_morpheme_freq'].dropna()
     for phon in phonation_order],
    positions=[0, 1, 2],
    widths=0.15,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color='black', linewidth=2),
    boxprops=dict(facecolor='lightblue', edgecolor='black', linewidth=1.5, alpha=0.8),
    whiskerprops=dict(color='black', linewidth=1.5),
    capprops=dict(color='black', linewidth=1.5)
)

# Set labels and title
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(phonation_order, fontsize=12)
ax.set_xlabel('Phonation Type', fontsize=13, fontweight='bold')
ax.set_ylabel('log frequency', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# Save figure
output_path = output_dir / "phonation_log_frequency_violin.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\nFigure saved to: {output_path}")

plt.show()

In [ ]:
df['prolific_id'].unique()

## VOT & F0 Distribution by Participant

In [ ]:
# F0 & VOT - 참가자별/나이별 (한 그래프, 4개 subplot, 세로 배치)
# 참가자(prolific_id)별로 Y값(첫번째: f0, 두번째: VOT) 기준 오름차순 정렬

def get_sorted_participants_by_y(df, y_col):
    # 각 참가자별 y_col의 평균값 기준 오름차순 정렬
    means = df.groupby('prolific_id')[y_col].mean().sort_values()
    return means.index.tolist()

participants_sorted_vot = get_sorted_participants_by_y(df, 'vot')
participants_sorted_f0 = get_sorted_participants_by_y(df, 'f0')
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# VOT by participant (vot 평균 오름차순)
sns.boxplot(data=df, x='prolific_id', y='vot', hue='prolific_id', order=participants_sorted_vot, palette='viridis', ax=axes[0], legend=False)
axes[0].set_title('VOT Distribution (by participant, sorted by VOT)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Participant', fontsize=12)
axes[0].set_ylabel('VOT (ms)', fontsize=12)
axes[0].set_ylim(0, 200)
axes[0].tick_params(axis='x', rotation=90)

# F0 by participant (f0 평균 오름차순)
sns.boxplot(data=df, x='prolific_id', y='f0', hue='prolific_id', order=participants_sorted_f0, palette='mako', ax=axes[1], legend=False)
axes[1].set_title('F0 Distribution (by participant, sorted by F0)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Participant', fontsize=12)
axes[1].set_ylabel('F0 (Hz)', fontsize=12)
axes[1].set_ylim(0, 400)
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "f0-and-vot-by-participant.png", dpi=300, bbox_inches='tight')
plt.show()

## Scaled VOT & F0 by Phonation and Gender

In [ ]:
# 2-2. Scaled VOT/F0 분포 (female vs male, 2x2 subplot, gender=col, scaled_vot/normed_f0=row, 좌우 y축 범위 일치)

fig, axes = plt.subplots(2, 2, figsize=(5, 6))  # CELL 16과 동일한 플롯 크기

# 각 row별로 박스플롯의 IQR(박스+수염)만 커버하는 y축 범위 계산 함수
def get_boxplot_range(data, y, x, order):
    q1 = data.groupby(x)[y].quantile(0.25).reindex(order)
    q3 = data.groupby(x)[y].quantile(0.75).reindex(order)
    iqr = q3 - q1
    whisker_low = (q1 - 1.5 * iqr).min()
    whisker_high = (q3 + 1.5 * iqr).max()
    return whisker_low, whisker_high

# 위: scaled_vot (좌: Female, 우: Male)
vot_low_f, vot_high_f = get_boxplot_range(df[df['gender']=='Female'], 'scaled_vot', 'phonation', phonation_order)
vot_low_m, vot_high_m = get_boxplot_range(df[df['gender']=='Male'], 'scaled_vot', 'phonation', phonation_order)
vot_min = min(vot_low_f, vot_low_m)
vot_max = max(vot_high_f, vot_high_m)

sns.boxplot(
    data=df[df['gender']=='Female'],
    x='phonation', y='scaled_vot', hue='phonation', order=phonation_order,
    palette=[phonation_colors[p] for p in phonation_order],
    ax=axes[0,0],
    width=0.5,  # CELL 16과 동일한 박스 폭
    dodge=False,
    legend=False,
    boxprops=dict(linewidth=1),
    medianprops=dict(linewidth=1.2),
    showcaps=True,
    flierprops=dict(marker='o', markersize=2),
    showfliers=False
    )
axes[0,0].set_title('Scaled VOT (Female)', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Phonation Type', fontsize=10)
axes[0,0].set_ylabel('Scaled VOT (word duration)', fontsize=10)
axes[0,0].set_ylim(0, 0.5)
axes[0,0].grid(True, alpha=0.3, axis='y')

sns.boxplot(
    data=df[df['gender']=='Male'],
    x='phonation', y='scaled_vot', hue='phonation', order=phonation_order,
    palette=[phonation_colors[p] for p in phonation_order],
    ax=axes[0,1],
    width=0.5,  # CELL 16과 동일한 박스 폭
    dodge=False,
    legend=False,
    boxprops=dict(linewidth=1),
    medianprops=dict(linewidth=1.2),
    showcaps=True,
    flierprops=dict(marker='o', markersize=2),
    showfliers=False
    )
axes[0,1].set_title('Scaled VOT (Male)', fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('Phonation Type', fontsize=10)
axes[0,1].set_ylim(0, 0.5)
axes[0,1].set_ylabel('Scaled VOT (word duration)', fontsize=10)
axes[0,1].grid(True, alpha=0.3, axis='y')

# 아래: normalized_f0 (좌: Female, 우: Male)
f0_low_f, f0_high_f = get_boxplot_range(df[df['gender']=='Female'], 'normed_f0', 'phonation', phonation_order)
f0_low_m, f0_high_m = get_boxplot_range(df[df['gender']=='Male'], 'normed_f0', 'phonation', phonation_order)
f0_min = min(f0_low_f, f0_low_m)
f0_max = max(f0_high_f, f0_high_m)

sns.boxplot(
    data=df[df['gender']=='Female'],
    x='phonation', y='normed_f0', hue='phonation', order=phonation_order,
    palette=[phonation_colors[p] for p in phonation_order],
    ax=axes[1,0],
    width=0.5,  # CELL 16과 동일한 박스 폭
    dodge=False,
    legend=False,
    boxprops=dict(linewidth=1),
    medianprops=dict(linewidth=1.2),
    showcaps=True,
    flierprops=dict(marker='o', markersize=2),
    showfliers=False
    )
axes[1,0].set_title('Normalized F0 (Female)', fontsize=11, fontweight='bold')
axes[1,0].set_ylim(-3, 3)
axes[1,0].set_xlabel('Phonation Type', fontsize=10)
axes[1,0].set_ylabel('Normalized F0 (z-score)', fontsize=10)
axes[1,0].grid(True, alpha=0.3, axis='y')

sns.boxplot(
    data=df[df['gender']=='Male'],
    x='phonation', y='normed_f0', hue='phonation', order=phonation_order,
    palette=[phonation_colors[p] for p in phonation_order],
    ax=axes[1,1],
    width=0.5,  # CELL 16과 동일한 박스 폭
    dodge=False,
    legend=False,
    boxprops=dict(linewidth=1),
    medianprops=dict(linewidth=1.2),
    showcaps=True,
    flierprops=dict(marker='o', markersize=2),
    showfliers=False
    )
axes[1,1].set_title('Normalized F0 (Male)', fontsize=11, fontweight='bold')
axes[1,1].set_ylim(-3, 3)
axes[1,1].set_xlabel('Phonation Type', fontsize=10)
axes[1,1].set_ylabel('Normalized F0 (z-score)', fontsize=10)
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout(pad=1.0, w_pad=0.5, h_pad=0.5)  # CELL 16과 동일한 간격 설정
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "scaled-vot-and-normed-f0-by-phonation-and-gender.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
df

## VOT & F0 by Phonation and Place of Articulation

In [ ]:
# raw VOT/F0 분포 (Phonation별, 박스플랏, 2행 3열 서브플롯)
import matplotlib.pyplot as plt
import seaborn as sns

# POA 컬럼을 소문자로 변환하여 색상 매핑이 정확하게 되도록
df['poa'] = df['poa'].str.lower()

poa_labels = ['labial', 'coronal', 'dorsal']
poa_order = poa_labels  # x축 순서
# Colorblind-friendly palette (Wong 2011) - phonation_colors와 일관성 있게 수정
poa_palette = {'labial': '#CC79A7', 'coronal': '#9467BD', 'dorsal': '#999999'}  # dark pink, purple, gray

# Phonation 레이블 매핑
phonation_label_map = {
    'lenis': 'Lenis',
    'fortis': 'Fortis',
    'aspirated': 'Aspirated'
}

y_vars = ['vot', 'f0']
y_labels = ['VOT (ms)', 'F0 (Hz)']

fig, axes = plt.subplots(2, 3, figsize=(7, 6), sharey='row')
for col, phonation in enumerate(phonation_order):
    for row, (yvar, ylabel) in enumerate(zip(y_vars, y_labels)):
        data = df[df['phonation'] == phonation]
        sns.boxplot(
            data=data,
            x='poa',
            y=yvar,
            hue='poa',
            order=poa_order,
            hue_order=poa_order,
            palette=poa_palette,
            ax=axes[row, col],
            width=0.5,
            dodge=False,
            legend=False,
            boxprops=dict(linewidth=1, edgecolor='black'),
            medianprops=dict(linewidth=1.5, color='black'),
            whiskerprops=dict(linewidth=1, color='black'),
            capprops=dict(linewidth=1, color='black'),
            showcaps=True,
            flierprops=dict(marker='o', markersize=2, markeredgecolor='black'),
            showfliers=False
        )
        # 제목 설정 - phonation 타입으로
        if row == 0:
            axes[row, col].set_title(f"{phonation_label_map[phonation]}", fontsize=11, fontweight='bold')
        axes[row, col].set_xlabel('POA', fontsize=9)
        if col == 0:
            axes[row, col].set_ylabel(ylabel, fontsize=9)
        else:
            axes[row, col].set_ylabel('')
        axes[row, col].set_xticks([0, 1, 2])
        axes[row, col].set_xticklabels([p.capitalize() for p in poa_order], fontsize=8)
        axes[row, col].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "vot-and-f0-by-phonation-and-poa.png", dpi=300, bbox_inches='tight')
plt.show()

## Age Effects on Acoustic Measures by Phonation Type

In [ ]:
# Age vs Scaled VOT and Normalized F0 - Scatter plot + Smooth curve (by phonation type, no gender split)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot for each dependent variable
for idx, (yvar, ylabel, ylim) in enumerate([
    ('scaled_vot', 'Scaled VOT (word duration)', (0, 0.4)),
    ('normed_f0', 'Normalized F0 (z-score)', (-2, 2))
]):
    ax = axes[idx]
    
    # Plot smooth curves with confidence bands for each phonation type
    for phonation in phonation_order:
        df_phon = df[df['phonation'] == phonation].dropna(subset=['age', yvar]).copy()
        
        # Sort by age for smooth curve
        df_sorted = df_phon.sort_values('age')
        
        # 데이터가 충분하면 smooth curve 그리기
        if len(df_sorted) > 10:
            # Age bins로 평균 계산 (moving average 효과)
            age_bins = np.arange(df_sorted['age'].min(), df_sorted['age'].max(), 5)
            bin_means = []
            bin_stds = []
            bin_ages = []
            
            for i in range(len(age_bins) - 1):
                mask = (df_sorted['age'] >= age_bins[i]) & (df_sorted['age'] < age_bins[i+1])
                if mask.sum() >= 3:  # 최소 3개 데이터포인트
                    bin_ages.append(age_bins[i] + 2.5)
                    bin_means.append(df_sorted[mask][yvar].mean())
                    bin_stds.append(df_sorted[mask][yvar].std())
            
            if len(bin_ages) > 3:
                # Smooth curve with polynomial fit (numpy only)
                try:
                    # 2차 또는 3차 다항식 피팅
                    degree = min(3, len(bin_ages) - 1)
                    coeffs = np.polyfit(bin_ages, bin_means, degree)
                    coeffs_upper = np.polyfit(bin_ages, [m + s for m, s in zip(bin_means, bin_stds)], degree)
                    coeffs_lower = np.polyfit(bin_ages, [m - s for m, s in zip(bin_means, bin_stds)], degree)
                    
                    age_smooth = np.linspace(min(bin_ages), max(bin_ages), 200)
                    y_smooth = np.polyval(coeffs, age_smooth)
                    y_upper = np.polyval(coeffs_upper, age_smooth)
                    y_lower = np.polyval(coeffs_lower, age_smooth)
                    
                    ax.fill_between(age_smooth, y_lower, y_upper, 
                                   color=phonation_colors[phonation], alpha=0.2)
                    ax.plot(age_smooth, y_smooth, 
                           color=phonation_colors[phonation], linewidth=3, 
                           label=phonation_label_map[phonation] if idx == 0 else None, alpha=0.9)
                except:
                    # Fallback to simple line
                    ax.plot(bin_ages, bin_means, 
                           color=phonation_colors[phonation], linewidth=3, 
                           label=phonation_label_map[phonation] if idx == 0 else None, alpha=0.9, marker='o')
    
    ax.set_title(f'Age vs {ylabel}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Age', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    if idx == 0:
        ax.legend(title='Phonation Type', fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(ylim)
    if yvar == 'normed_f0':
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1.2, alpha=0.5)


plt.tight_layout()
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "x-age-y-acoustics-hue-phonation-confidence-bands.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nData distribution by phonation type:")
for phonation in phonation_order:
    count = len(df[df['phonation'] == phonation].dropna(subset=['age', 'normed_f0', 'scaled_vot']))
    print(f"{phonation_label_map[phonation].capitalize()}: {count} observations")

## Age Effects on Acoustic Measures by Phonation Type and Gender

In [ ]:
# Age vs Scaled VOT and Normalized F0 - Scatter plot + Smooth curve (gender split comparison)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
df['scaled_vot'] = df['vot'] / df['word_duration']

# Plot for each combination of dependent variable and gender
for row_idx, (yvar, ylabel, ylim) in enumerate([
    ('scaled_vot', 'Scaled VOT (word duration)', (0, 0.5)),
    ('normed_f0', 'Normalized F0 (z-score)', (-2, 2))
]):
    for col_idx, gender in enumerate(['Female', 'Male']):
        ax = axes[row_idx, col_idx]
        
        # Filter by gender
        df_gender = df[df['gender'] == gender].copy()
        
        # Plot smooth curves with confidence bands for each phonation type
        for phonation in phonation_order:
            df_phon = df_gender[df_gender['phonation'] == phonation].dropna(subset=['age', yvar]).copy()
            
            # Sort by age for smooth curve
            df_sorted = df_phon.sort_values('age')
            
            # 데이터가 충분하면 smooth curve 그리기
            if len(df_sorted) > 10:
                # Age bins로 평균 계산 (moving average 효과)
                age_bins = np.arange(df_sorted['age'].min(), df_sorted['age'].max(), 5)
                bin_means = []
                bin_stds = []
                bin_ages = []
                
                for i in range(len(age_bins) - 1):
                    mask = (df_sorted['age'] >= age_bins[i]) & (df_sorted['age'] < age_bins[i+1])
                    if mask.sum() >= 3:  # 최소 3개 데이터포인트
                        bin_ages.append(age_bins[i] + 2.5)
                        bin_means.append(df_sorted[mask][yvar].mean())
                        bin_stds.append(df_sorted[mask][yvar].std())
                
                if len(bin_ages) > 3:
                    # Smooth curve with polynomial fit (numpy only)
                    try:
                        # 2차 또는 3차 다항식 피팅
                        degree = min(3, len(bin_ages) - 1)
                        coeffs = np.polyfit(bin_ages, bin_means, degree)
                        coeffs_upper = np.polyfit(bin_ages, [m + s for m, s in zip(bin_means, bin_stds)], degree)
                        coeffs_lower = np.polyfit(bin_ages, [m - s for m, s in zip(bin_means, bin_stds)], degree)
                        
                        age_smooth = np.linspace(min(bin_ages), max(bin_ages), 200)
                        y_smooth = np.polyval(coeffs, age_smooth)
                        y_upper = np.polyval(coeffs_upper, age_smooth)
                        y_lower = np.polyval(coeffs_lower, age_smooth)
                        
                        ax.fill_between(age_smooth, y_lower, y_upper, 
                                       color=phonation_colors[phonation], alpha=0.2)
                        ax.plot(age_smooth, y_smooth, 
                               color=phonation_colors[phonation], linewidth=3, 
                               label=phonation_label_map[phonation] if (row_idx == 0 and col_idx == 0) else None, alpha=0.9)
                    except:
                        # Fallback to simple line
                        ax.plot(bin_ages, bin_means, 
                               color=phonation_colors[phonation], linewidth=3, 
                               label=phonation_label_map[phonation] if (row_idx == 0 and col_idx == 0) else None, alpha=0.9, marker='o')
        
        ax.set_title(f'Age vs {ylabel} - {gender}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Age', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        if row_idx == 0 and col_idx == 0:
            ax.legend(title='Phonation Type', fontsize=10, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(ylim)
        if yvar == 'normed_f0':
            ax.axhline(y=0, color='gray', linestyle='--', linewidth=1.2, alpha=0.5)

plt.tight_layout()
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "x-age-y-acoustics-hue-phonation-panel-gender-confidence-bands-test.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nData distribution by phonation type and gender:")
for phonation in phonation_order:
    print(f"\n{phonation_label_map[phonation].capitalize()}:")
    for gender in ['Male', 'Female']:
        count = len(df[(df['phonation'] == phonation) & (df['gender'] == gender)].dropna(subset=['age', 'normed_f0', 'scaled_vot']))
        print(f"  {gender}: {count} observations")

## Age Effects with Data Points and Trend Lines

In [ ]:
# Age effects - 3x4 grid: rows=phonation, cols=gender×measure
# Each phonation type gets its own row, making it easy to compare patterns across measures and genders

fig, axes = plt.subplots(3, 4, figsize=(20, 12))

# Define measure configurations
measures = [
    ('scaled_vot', 'Scaled VOT\n(word duration)', (0, 0.5)),
    ('normed_f0', 'Normalized F0\n(z-score)', (-2, 2))
]

genders = ['Female', 'Male']

# Iterate through phonation types (rows)
for row_idx, phonation in enumerate(phonation_order):
    
    # Iterate through gender × measure combinations (columns)
    for col_idx, (gender, (yvar, ylabel, ylim)) in enumerate(
        [(g, m) for g in genders for m in measures]
    ):
        ax = axes[row_idx, col_idx]
        
        # Filter data
        df_subset = df[(df['phonation'] == phonation) & (df['gender'] == gender)].dropna(subset=['age', yvar]).copy()
        
        if len(df_subset) > 0:
            # Plot scatter points
            ax.scatter(
                df_subset['age'], 
                df_subset[yvar],
                c=phonation_colors[phonation],
                marker=phonation_markers[phonation],
                s=25,
                alpha=0.4,
                edgecolors='none'
            )
            
            # Calculate and plot trend line
            df_sorted = df_subset.sort_values('age')
            
            if len(df_sorted) > 10:
                age_bins = np.arange(df_sorted['age'].min(), df_sorted['age'].max(), 5)
                bin_means = []
                bin_ages = []
                
                for i in range(len(age_bins) - 1):
                    mask = (df_sorted['age'] >= age_bins[i]) & (df_sorted['age'] < age_bins[i+1])
                    if mask.sum() >= 3:
                        bin_ages.append(age_bins[i] + 2.5)
                        bin_means.append(df_sorted[mask][yvar].mean())
                
                if len(bin_ages) > 3:
                    try:
                        degree = min(3, len(bin_ages) - 1)
                        coeffs = np.polyfit(bin_ages, bin_means, degree)
                        age_smooth = np.linspace(min(bin_ages), max(bin_ages), 200)
                        y_smooth = np.polyval(coeffs, age_smooth)
                        
                        ax.plot(age_smooth, y_smooth, 
                               color=phonation_colors[phonation], 
                               linewidth=2.5, 
                               alpha=0.9)
                    except:
                        ax.plot(bin_ages, bin_means, 
                               color=phonation_colors[phonation], 
                               linewidth=2.5,
                               alpha=0.9,
                               marker='o')
        
        # Set title (phonation name on first column only)
        if col_idx == 0:
            ax.set_title(f'{phonation_label_map[phonation]} | {gender}\n{ylabel}', 
                        fontsize=10, fontweight='bold', pad=8)
        else:
            measure_name = ylabel.split('\n')[0]  # Get first line only
            ax.set_title(f'{gender} | {measure_name}', 
                        fontsize=10, fontweight='bold', pad=8)
        
        # Labels
        if row_idx == 2:  # Bottom row
            ax.set_xlabel('Age', fontsize=9)
        else:
            ax.set_xlabel('')
            
        if col_idx == 0:  # First column
            ax.set_ylabel(ylabel, fontsize=9)
        else:
            ax.set_ylabel('')
        
        ax.grid(True, alpha=0.3)
        ax.set_ylim(ylim)
        
        # Add horizontal line at 0 for F0 plots
        if yvar == 'normed_f0':
            ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        
        # Data count annotation
        n_points = len(df_subset)
        ax.text(0.02, 0.98, f'n={n_points}', 
               transform=ax.transAxes, 
               fontsize=8, 
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'))

plt.tight_layout()
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "age-effects-by-phonation-3x4-grid.png", dpi=300, bbox_inches='tight')
plt.show()

print("\nGrid structure: 3 rows (phonation types) × 4 columns (Female-VOT, Female-F0, Male-VOT, Male-F0)")
print("Each subplot shows age trends for a specific phonation × gender × measure combination")